# 03 — Modelado de elasticidad

**Proyecto:** Análisis de Elasticidad de Demanda en Delivery de Comida (v2)
**Fase:** 3 — Modelado de elasticidad

Objetivos de este notebook:
1. Ajustar el modelo base log-log (elasticidad precio simple)
2. Ajustar un modelo log-log con variables de control (promoción, homepage, aumento de precio, efectos fijos por categoría y tipo de centro)
3. Ajustar una especificación alternativa semi-log, para comparación
4. Comparar las tres especificaciones con AIC/BIC y seleccionar la más adecuada
5. Calcular un intervalo de confianza por bootstrap para la elasticidad estimada


In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from modeling import (
    modelo_log_log_base,
    modelo_log_log_controles,
    modelo_semilog,
    tabla_comparacion_modelos,
    bootstrap_elasticidad,
)

pd.set_option('display.max_columns', None)

In [3]:
from pathlib import Path

from IPython.display import display

REPORTS_DIR = Path('../reports')
NOTEBOOK_OUTPUTS_DIR = REPORTS_DIR / 'notebook_outputs'
NOTEBOOK_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)


def guardar_texto(nombre_archivo: str, contenido: str) -> Path:
    ruta = NOTEBOOK_OUTPUTS_DIR / nombre_archivo
    ruta.write_text(str(contenido), encoding='utf-8')
    return ruta


def guardar_summary(nombre_archivo: str, resultado_modelo) -> tuple[Path, str]:
    texto = resultado_modelo.summary().as_text()
    ruta = guardar_texto(nombre_archivo, texto)
    return ruta, texto


def guardar_tabla(nombre_base: str, tabla: pd.DataFrame) -> tuple[Path, Path]:
    ruta_txt = NOTEBOOK_OUTPUTS_DIR / f'{nombre_base}.txt'
    ruta_csv = NOTEBOOK_OUTPUTS_DIR / f'{nombre_base}.csv'
    tabla.to_csv(ruta_csv, index=False)
    ruta_txt.write_text(tabla.to_string(index=False), encoding='utf-8')
    return ruta_txt, ruta_csv

## 1. Cargar el dataset procesado (Fase 2)

In [4]:
df = pd.read_csv('../data/processed/demanda_limpia.csv')
ruta_shape = guardar_texto('03_modelado_elasticidad_df_shape.txt', f'Shape del dataset: {df.shape}')
print(f'Shape del dataset: {df.shape}')
print(f'Guardado en {ruta_shape}')

Shape del dataset: (456548, 20)
Guardado en ..\reports\notebook_outputs\03_modelado_elasticidad_df_shape.txt


## 2. Modelo base: log-log simple

`log_num_orders ~ log_checkout_price`

El coeficiente de `log_checkout_price` se interpreta directamente como la
elasticidad precio de la demanda: el cambio porcentual esperado en el número
de pedidos ante un cambio de 1% en el precio.

In [5]:
modelo_base = modelo_log_log_base(df)
ruta_resumen_base, resumen_base = guardar_summary('03_modelado_elasticidad_modelo_base_summary.txt', modelo_base)
print(resumen_base)
print(f'Summary guardado en {ruta_resumen_base}')

                            OLS Regression Results                            
Dep. Variable:         log_num_orders   R-squared:                       0.142
Model:                            OLS   Adj. R-squared:                  0.142
Method:                 Least Squares   F-statistic:                 7.575e+04
Date:                Thu, 06 Aug 2026   Prob (F-statistic):               0.00
Time:                        11:02:21   Log-Likelihood:            -6.9721e+05
No. Observations:              456548   AIC:                         1.394e+06
Df Residuals:                  456546   BIC:                         1.394e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.2410      0

**Lectura del resultado:** el coeficiente de `log_checkout_price` (fila
correspondiente en la tabla de arriba) es la elasticidad estimada. Un
coeficiente de, por ejemplo, -1.2 significaría que un aumento de 1% en el
precio se asocia con una caída de 1.2% en el número de pedidos (demanda
elástica). Documentar el valor exacto obtenido, su error estándar y su
significancia (columna P>|t|) en la bitácora una vez corrida esta celda.

## 3. Modelo con variables de control

Se agregan: `emailer_for_promotion`, `homepage_featured`, `hubo_aumento`
(hallazgo de la Fase 2), y efectos fijos por categoría de comida
(`C(category)`) y tipo de centro (`C(center_type)`).

**Por qué agregar efectos fijos:** distintas categorías de comida y tipos de
centro pueden tener niveles de demanda base muy distintos, independientes del
precio. Sin controlar por esto, el coeficiente de precio podría capturar
parte de esas diferencias en vez de un efecto de precio puro.

In [6]:
modelo_controles = modelo_log_log_controles(df)
ruta_resumen_controles, resumen_controles = guardar_summary('03_modelado_elasticidad_modelo_controles_summary.txt', modelo_controles)
print(resumen_controles)
print(f'Summary guardado en {ruta_resumen_controles}')

                            OLS Regression Results                            
Dep. Variable:         log_num_orders   R-squared:                       0.531
Model:                            OLS   Adj. R-squared:                  0.531
Method:                 Least Squares   F-statistic:                 2.724e+04
Date:                Thu, 06 Aug 2026   Prob (F-statistic):               0.00
Time:                        11:02:32   Log-Likelihood:            -5.5924e+05
No. Observations:              456548   AIC:                         1.119e+06
Df Residuals:                  456528   BIC:                         1.119e+06
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

**Comparar con el modelo base:** ¿el coeficiente de `log_checkout_price`
cambió mucho al agregar controles? Si cambia poco, es una señal de robustez
del resultado. Si cambia mucho, sugiere que el modelo base estaba
confundiendo el efecto de precio con diferencias entre categorías/centros.

## 4. Especificación alternativa: semi-log

`log_num_orders ~ checkout_price` (precio en nivel, no logarítmico), con los
mismos controles. Aquí el coeficiente de precio ya no es una elasticidad
directa, sino un efecto semi-elástico (cambio porcentual en la demanda ante
un aumento de 1 unidad monetaria en el precio).

In [7]:
modelo_semi = modelo_semilog(df)
ruta_resumen_semi, resumen_semi = guardar_summary('03_modelado_elasticidad_modelo_semilog_summary.txt', modelo_semi)
print(resumen_semi)
print(f'Summary guardado en {ruta_resumen_semi}')

                            OLS Regression Results                            
Dep. Variable:         log_num_orders   R-squared:                       0.538
Model:                            OLS   Adj. R-squared:                  0.538
Method:                 Least Squares   F-statistic:                 2.795e+04
Date:                Thu, 06 Aug 2026   Prob (F-statistic):               0.00
Time:                        11:02:47   Log-Likelihood:            -5.5614e+05
No. Observations:              456548   AIC:                         1.112e+06
Df Residuals:                  456528   BIC:                         1.113e+06
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

## 5. Comparación de especificaciones (AIC / BIC)

AIC y BIC penalizan la complejidad del modelo (número de parámetros) además
de premiar el ajuste. Un valor más bajo indica mejor balance entre ajuste y
parsimonia. Se usan para decidir cuál especificación se reporta como
resultado principal del proyecto.

In [8]:
modelos = {
    'log-log base': modelo_base,
    'log-log con controles': modelo_controles,
    'semi-log con controles': modelo_semi,
}

tabla_modelos = tabla_comparacion_modelos(modelos)
ruta_tabla_modelos, ruta_csv_modelos = guardar_tabla('03_modelado_elasticidad_comparacion_modelos', tabla_modelos)
print(f'Tabla guardada en {ruta_csv_modelos} y {ruta_tabla_modelos}')
tabla_modelos

Tabla guardada en ..\reports\notebook_outputs\03_modelado_elasticidad_comparacion_modelos.csv y ..\reports\notebook_outputs\03_modelado_elasticidad_comparacion_modelos.txt


,modelo,aic,bic,r2_ajustado,n_observaciones
0,semi-log con controles,1.112317e+06,1.112537e+06,0.537668,456548
1,log-log con controles,1.118513e+06,1.118733e+06,0.531351,456548
2,log-log base,1.394427e+06,1.394449e+06,0.142310,456548


**Criterio de selección:** se elige el modelo con menor AIC/BIC entre las
especificaciones comparables (el modelo base no es directamente comparable
por AIC/BIC contra los otros dos porque tiene un conjunto de variables
distinto — la comparación relevante es entre 'log-log con controles' y
'semi-log con controles'). Documentar en la bitácora cuál resultó preferida
y por qué margen.

## 6. Intervalo de confianza por bootstrap para la elasticidad (COMENTADO - EJECUTAR EN VM)

Se remuestrea el dataset con reemplazo 1,000 veces, se reajusta el modelo
base en cada remuestra, y se guarda el coeficiente de `log_checkout_price`
en cada iteración. El intervalo de confianza al 95% se construye con los
percentiles 2.5 y 97.5 de esa distribución de coeficientes.

**Por qué esto importa:** el modelo base ya entrega un error estándar y un
intervalo de confianza analítico (visibles en el `summary()` de la sección
2), pero ese intervalo depende de los supuestos clásicos de OLS (normalidad
de errores, homocedasticidad), que se van a poner a prueba recién en la
Fase 4. El bootstrap es una forma no paramétrica de estimar la incertidumbre,
que no depende de esos supuestos, y sirve como verificación cruzada del
intervalo analítico.

**NOTA IMPORTANTE - CELDA COMENTADA:** Esta computación (1000 iteraciones sobre 456,548 filas)
tarda varios minutos. SE HA MOVIDO AL SCRIPT DE VM: `scripts/vm_computaciones_pesadas.py`
Ese script se ejecuta en una VM con GPU 100% dedicado + 32GB RAM y produce:
`reports/notebook_outputs/bootstrap_quantile_results.npy` (para cuantiles) y
`reports/notebook_outputs/bootstrap_panel_fe_results.npy` (para panel FE clusterizado).

El resultado del bootstrap simple OLS (modelo base) se cargará desde ese archivo .npy
en el pipeline principal (`scripts/reproducir_resultados_etapa3.py`), NO se ejecuta aquí.

**Nota de tiempo de ejecución:** con ~450,000 filas y 1,000 iteraciones, esta
celda puede tardar varios minutos. Si el tiempo es una limitante, reducir
`n_iteraciones` a 200-300 como primera aproximación.

In [ ]:
# CELDA COMENTADA - Bootstrap simple OLS movido a VM (scripts/vm_computaciones_pesadas.py)
# Ver sección 6 arriba: esta computación se ejecuta en VM aparte y los resultados
# se cargan desde reports/notebook_outputs/bootstrap_quantile_results.npy (para cuantiles)
# y bootstrap_panel_fe_results.npy (para panel FE) en el pipeline principal.
#
# Código original (para referencia):
# resultado_bootstrap = bootstrap_elasticidad(df, n_iteraciones=1000)
# texto_bootstrap = (
#     f"Elasticidad media (bootstrap): {resultado_bootstrap['elasticidad_media']:.4f}\n"
#     f"Desviación estándar         : {resultado_bootstrap['elasticidad_std']:.4f}\n"
#     f"IC 95%                      : [{resultado_bootstrap['ic_95_inferior']:.4f}, "
#     f"{resultado_bootstrap['ic_95_superior']:.4f}]"
# )
# ruta_bootstrap = guardar_texto('03_modelado_elasticidad_bootstrap.txt', texto_bootstrap)
# print(texto_bootstrap)
# print(f'Resultados guardados en {ruta_bootstrap}')

print("CELDA COMENTADA: bootstrap_elasticidad (1000 iter) se ejecuta en VM aparte.")
print("Ver: scripts/vm_computaciones_pesadas.py")
print("Resultados se cargarán desde reports/notebook_outputs/*.npy en pipeline principal.")

## 7. Guardar resultados de la Fase 3

Se guarda la tabla comparativa de modelos, para poder referenciarla
directamente en el informe final sin tener que re-ejecutar el notebook.

In [24]:
ruta_comparacion = guardar_texto('03_modelado_elasticidad_comparacion_modelos_generacion.txt', 'Guardado en reports/comparacion_modelos.csv y reports/notebook_outputs/03_modelado_elasticidad_comparacion_modelos.txt')
tabla_modelos.to_csv('../reports/comparacion_modelos.csv', index=False)
print(f'Guardado en reports/comparacion_modelos.csv y {ruta_comparacion}')

Guardado en reports/comparacion_modelos.csv y ..\reports\notebook_outputs\03_modelado_elasticidad_comparacion_modelos_generacion.txt


## 8. Especificación flexible: Splines en log-precio (RCS)

Se implementa un modelo log-log con Restricted Cubic Splines (RCS) en log_checkout_price para testear no-linealidad en la relación precio-demanda. Se usan 4 nudos en percentiles 5, 35, 65, 95 de log_checkout_price (df=4).

El modelo incluye los mismos controles que la especificación principal: emailer_for_promotion, homepage_featured, hubo_aumento, C(category), C(center_type).

In [ ]:
from src.modeling import modelo_log_log_rcs, test_wald_no_linealidad_rcs, elasticidad_puntual_rcs
import numpy as np

modelo_rcs, spline_design, nudos = modelo_log_log_rcs(df, df_splines=4, percentiles=(5, 35, 65, 95))
print(f"Nudos (log-precio): {nudos}")
print(f"Nudos (precio original): {np.exp(nudos)}")
print(modelo_rcs.summary())

In [ ]:
# Test de Wald para no-linealidad: H0 = coeficientes términos no lineales = 0
spline_cols = [f"rcs_{i}" for i in range(spline_design.shape[1])]
wald_result = test_wald_no_linealidad_rcs(modelo_rcs, spline_cols)
print(f"Test Wald no-linealidad:")
print(f"  F-estadístico: {wald_result["estadistico_F"]:.4f}")
print(f"  p-valor: {wald_result["p_valor"]:.6f}")
print(f"  Grados libertad: {wald_result["grados_libertad"]}")
print(f"  Rechaza H0 (α=0.05): {wald_result["rechaza_H0_05"]}")

In [ ]:
# Elasticidad puntual a lo largo de la distribución de precios
precio_min = df["checkout_price"].min()
precio_max = df["checkout_price"].max()
precio_eval = np.linspace(precio_min, precio_max, 100)
elasticidades = elasticidad_puntual_rcs(modelo_rcs, spline_design, precio_eval)

import pandas as pd
df_elasticidad = pd.DataFrame({"precio": precio_eval, "elasticidad": elasticidades})
print(df_elasticidad.head(10))
print(f"
Elasticidad en precio medio ({df["checkout_price"].mean():.2f}): {np.interp(df["checkout_price"].mean(), precio_eval, elasticidades):.4f}")

In [ ]:
# Comparación AIC/BIC con modelos previos
from src.modeling import tabla_comparacion_modelos

modelos_todos = {
    "log-log base": modelo_base,
    "log-log con controles": modelo_controles,
    "semi-log con controles": modelo_semi,
    "log-log RCS (4 nudos) con controles": modelo_rcs,
}

tabla_completa = tabla_comparacion_modelos(modelos_todos)
print(tabla_completa.to_string(index=False))

ruta_tabla_rcs, ruta_csv_rcs = guardar_tabla("03_modelado_elasticidad_comparacion_rcs", tabla_completa)
print(f"Tabla guardada en {ruta_csv_rcs} y {ruta_tabla_rcs}")

In [ ]:
# Gráfico: Elasticidad puntual vs Precio
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(precio_eval, elasticidades, "b-", linewidth=2, label="Elasticidad puntual (RCS)")
plt.axhline(y=modelo_controles.params["log_checkout_price"], color="r", linestyle="--", 
            label=f"Elasticidad log-log constante: {modelo_controles.params["log_checkout_price"]:.4f}")
plt.axvline(x=df["checkout_price"].mean(), color="gray", linestyle=":", 
            label=f"Precio medio: {df["checkout_price"].mean():.1f}")
plt.xlabel("Precio de checkout (S/)")
plt.ylabel("Elasticidad precio-demanda")
plt.title("Elasticidad puntual vs Precio - Modelo RCS (4 nudos)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

ruta_fig = Path("../reports/figuras/elasticidad_puntual_rcs.png")
plt.savefig(ruta_fig, dpi=300, bbox_inches="tight")
print(f"Figura guardada en {ruta_fig}")
plt.show()

## Próximos pasos (Fase 4 — validación de supuestos)
- Test de heterocedasticidad (Breusch-Pagan) sobre el modelo seleccionado
- VIF para multicolinealidad entre las variables de control
- Normalidad de residuos (Q-Q plot)
- Discusión de limitaciones de identificación causal

## Próximos pasos (Fase 4 — validación de supuestos)
- Test de heterocedasticidad (Breusch-Pagan) sobre el modelo seleccionado
- VIF para multicolinealidad entre las variables de control
- Normalidad de residuos (Q-Q plot)
- Discusión de limitaciones de identificación causal

In [25]:
precio_promedio = df['checkout_price'].mean()
elasticidad_semilog_equivalente = modelo_semi.params['checkout_price'] * precio_promedio
texto_equivalencia = (
    f'Precio promedio: {precio_promedio:.2f}\n'
    f'Elasticidad equivalente (semi-log, evaluada en la media): {elasticidad_semilog_equivalente:.4f}'
)
ruta_equivalencia = guardar_texto('03_modelado_elasticidad_equivalencia_semilog.txt', texto_equivalencia)
print(texto_equivalencia)
print(f'Guardado en {ruta_equivalencia}')

Precio promedio: 332.24
Elasticidad equivalente (semi-log, evaluada en la media): -1.0624
Guardado en ..\reports\notebook_outputs\03_modelado_elasticidad_equivalencia_semilog.txt


In [26]:
ruta_resumen_controles_2, resumen_controles_2 = guardar_summary('03_modelado_elasticidad_modelo_controles_summary_repetido.txt', modelo_controles)
print(resumen_controles_2)
print(f'Summary guardado en {ruta_resumen_controles_2}')

                            OLS Regression Results                            
Dep. Variable:         log_num_orders   R-squared:                       0.531
Model:                            OLS   Adj. R-squared:                  0.531
Method:                 Least Squares   F-statistic:                 2.724e+04
Date:               dom, 02 Ago. 2026   Prob (F-statistic):               0.00
Time:                        14:19:25   Log-Likelihood:            -5.5924e+05
No. Observations:              456548   AIC:                         1.119e+06
Df Residuals:                  456528   BIC:                         1.119e+06
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [27]:
# Guarda todo el resumen en un archivo de texto en tu carpeta actual
with open("resumen_modelo.txt", "w") as archivo:
    archivo.write(str(modelo_controles.summary()))

In [28]:
resumen_descriptivo = df[['checkout_price', 'base_price', 'num_orders', 'variacion_precio']].describe()
ruta_descriptivo = guardar_texto('03_modelado_elasticidad_descriptivo_variables.txt', resumen_descriptivo.to_string())
print(resumen_descriptivo)
print(f'Descriptivo guardado en {ruta_descriptivo}')

       checkout_price     base_price     num_orders  variacion_precio
count   456548.000000  456548.000000  456548.000000     456548.000000
mean       332.238933     354.156627     261.872760         21.917695
std        152.939723     160.715914     395.922798         48.495857
min          2.970000      55.350000      13.000000       -185.300000
25%        228.950000     243.500000      54.000000         -1.000000
50%        296.820000     310.460000     136.000000          1.000000
75%        445.230000     458.870000     324.000000         16.520000
max        866.270000     866.270000   24299.000000        561.600000
Descriptivo guardado en ..\reports\notebook_outputs\03_modelado_elasticidad_descriptivo_variables.txt
